# Open-XTTS-Trainer: End-to-End 3-Stage Training Demo

This notebook demonstrates how to train or fine-tune all 3 stages of **XTTS v2**:
1. **Stage 2:** Text Tokenizer & Vocabulary Extension for New Scripts (e.g. Arabic, Hindi, Cyrillic)
2. **Stage 3:** Latent HiFi-GAN Vocoder (Extracting GPT latents & training generator + discriminator)
3. **Stage 1:** Discrete VAE (DVAE) Audio Tokenizer fine-tuning

## 1. Setup & Environment

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

# Check GPU availability
!nvidia-smi

# Install open-xtts-trainer and Idiap's Coqui TTS fork
!pip install -q git+https://github.com/idiap/coqui-ai-TTS.git
!git clone https://github.com/amitbhakt/open-xtts-trainer.git 2>/dev/null || (cd open-xtts-trainer && git pull)
%cd open-xtts-trainer
!pip install -q -e .

## 2. Stage 2: Extend Vocabulary for a New Language

In [ ]:
# Download official XTTS v2 vocab.json
!mkdir -p xtts_base
!wget -q -O xtts_base/vocab.json https://huggingface.co/coqui/XTTS-v2/raw/main/vocab.json

# Sample target language text (e.g. Dhivehi, Thai, or unsupported scripts)
with open('sample_target.txt', 'w', encoding='utf-8') as f:
    f.write('ދިވެހި ބަސް. สวัสดี. مرحبا.')

# Run vocabulary extender
!python xtts_trainer/stage2_gpt/vocab_extender.py \
    --base_vocab xtts_base/vocab.json \
    --target_text sample_target.txt \
    --output_vocab xtts_base/vocab_extended.json

## 3. Stage 3: Latent HiFi-GAN Vocoder Training

In [ ]:
# Create sample audio files for demo training
from scripts.run_mini_test import generate_synthetic_audio
generate_synthetic_audio('sample_audio', num_files=8, duration_sec=1.5)

# 1. Extract GPT Latents
!python xtts_trainer/stage3_hifigan/extract_latents.py \
    --audio_dir sample_audio \
    --output_dir extracted_latents \
    --device cuda

# 2. Train Latent HiFi-GAN Vocoder
!python xtts_trainer/stage3_hifigan/train.py \
    --audio_dir sample_audio \
    --latents_dir extracted_latents \
    --output_dir checkpoints/hifigan \
    --batch_size 4 \
    --epochs 10 \
    --device cuda

## 4. Stage 1: Discrete VAE (DVAE) Fine-Tuning

In [ ]:
# Download official dvae.pth checkpoint to fine-tune from
!wget -q -O xtts_base/dvae.pth https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth

# Run DVAE training/fine-tuning
!python xtts_trainer/stage1_dvae/train.py \
    --audio_dir sample_audio \
    --output_dir checkpoints/dvae \
    --continue_path xtts_base/dvae.pth \
    --batch_size 4 \
    --epochs 10 \
    --device cuda


## 5. View Checkpoints & Play Audio Results

In [ ]:
# 1. List saved model checkpoints
print('--- HiFi-GAN Vocoder Checkpoints ---')
!ls -lh checkpoints/hifigan/

print('\n--- DVAE Audio Tokenizer Checkpoints ---')
!ls -lh checkpoints/dvae/

# 2. Play sample audio directly in Colab
import IPython.display as ipd
ipd.Audio('sample_audio/sample_001.wav')
